<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session06-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 6: in-class discussion problem (2 of 3)

This is the block 2 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then fix and run the code below, discuss the outcome with your neighbour, and answer the quiz again.

## Not every protein looks like haemoglobin

Haemoglobin's β chain (80.8% helix, 0% strand) and myoglobin (79.7% helix, 0% strand) are almost
entirely α-helical. That might make it tempting to think that most proteins are mostly helix, but
the globins happen to be an all-α family.

Green fluorescent protein (GFP, PDB `1EMA`) is a famous counter-example: a barrel of β-strands
wrapped around a single central helix that carries the fluorophore. What are its percentages of
helix and strand?

**Fix and run.** The second cell below computes the composition of myoglobin first, as a reference,
and then of GFP. The GFP part was made by copying the myoglobin part, and the line marked `FIX`
still has a leftover from the copy. Run it once as it is: what does it say about GFP? Then fix the
line and run it again.

In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython"), ("pymol", "pymol-open-source-whl")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
from Bio.PDB import PDBList
from pymol import cmd                  # PyMOL as a Python library, no window needed

pdbl = PDBList(verbose=False)

# Myoglobin, the reference
cmd.load(pdbl.retrieve_pdb_file("1mbn", pdir=".", file_format="pdb"), "mb")
cmd.dss("mb")
ss_mb = []
cmd.iterate("mb and polymer and name CA", "ss_mb.append(ss)", space={"ss_mb": ss_mb})
print(f"myoglobin: {100 * ss_mb.count('H') / len(ss_mb):.1f}% helix, "
      f"{100 * ss_mb.count('S') / len(ss_mb):.1f}% strand")

# GFP, copied from the part above
cmd.load(pdbl.retrieve_pdb_file("1ema", pdir=".", file_format="pdb"), "gfp")
cmd.dss("gfp")
ss_gfp = []
cmd.iterate("mb and polymer and name CA", "ss_gfp.append(ss)", space={"ss_gfp": ss_gfp})  # FIX
print(f"GFP:       {100 * ss_gfp.count('H') / len(ss_gfp):.1f}% helix, "
      f"{100 * ss_gfp.count('S') / len(ss_gfp):.1f}% strand")

myoglobin: 79.7% helix, 0.0% strand


GFP:       79.7% helix, 0.0% strand


**If you finish early.**

1. How could you write the composition once, as a function, so that this mistake cannot happen?
2. Proline and glycine disrupt regular backbone geometry and rarely sit in the middle of helices or
   strands. What would you predict for a new sequence that is rich in both throughout?

**Discussion point.** As it was, the GFP part still read the labels of `mb`, so it printed
myoglobin's numbers a second time and GFP looked like another globin. A copied block of code is a
common source of exactly this mistake; a function `composition(name)` called twice avoids it.
Fixed (`"gfp and polymer and name CA"`), GFP comes out at **9.8% helix, 53.8% strand** (22 and 121
of 225 residues): almost the mirror image of the globins.

That is why secondary-structure prediction cannot rely on a fixed rule such as "assume mostly
helix": the answer depends on the protein family, and Session 7's methods have to *learn*
position-specific patterns from data. A sequence rich in proline and glycine throughout would be
predicted to be mostly coil, or disordered.